In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.metrics import mean_squared_error

import mlflow

mlflow.set_tracking_uri("sqlite:///mlflow.db")

mlflow.set_experiment("nyc-taxi-experiment")

MlflowException: Detected out-of-date database schema (found version 97727af70f4d, but expected bd07f7e963c5). Take a backup of your database, then run 'mlflow db upgrade <database_uri>' to migrate your database to the latest schema. NOTE: schema migration may result in database downtime - please consult your database's documentation for more detail.

In [ ]:
import pickle

In [2]:
! pip freeze | grep scikit-learn

scikit-learn==1.1.1


In [2]:
#!pip install pyarrow

In [23]:
# data = pd.read_parquet("data/yellow_tripdata_2022-01.parquet")
data = pd.read_parquet("https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-01.parquet")
data.shape

(76518, 20)

### Q1 Read the data for January. How many columns are there?

In [24]:
print(f"The data has {data.shape[1]} columns")

The data has 20 columns


In [25]:
# val = pd.read_parquet("data/yellow_tripdata_2022-02.parquet")
val = pd.read_parquet("https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-02.parquet")
val.head()

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge
0,2,2021-02-01 00:34:03,2021-02-01 00:51:58,N,1.0,130,205,5.0,3.66,14.0,0.5,0.5,10.00,0.0,None,0.3,25.30,1.0,1.0,0.00
1,2,2021-02-01 00:04:00,2021-02-01 00:10:30,N,1.0,152,244,1.0,1.10,6.5,0.5,0.5,0.00,0.0,None,0.3,7.80,2.0,1.0,0.00
2,2,2021-02-01 00:18:51,2021-02-01 00:34:06,N,1.0,152,48,1.0,4.93,16.5,0.5,0.5,0.00,0.0,None,0.3,20.55,2.0,1.0,2.75
3,2,2021-02-01 00:53:27,2021-02-01 01:11:41,N,1.0,152,241,1.0,6.70,21.0,0.5,0.5,0.00,0.0,None,0.3,22.30,2.0,1.0,0.00
4,2,2021-02-01 00:57:46,2021-02-01 01:06:44,N,1.0,75,42,1.0,1.89,8.5,0.5,0.5,2.45,0.0,None,0.3,12.25,1.0,1.0,0.00


In [26]:
data.head()

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge
0,2,2021-01-01 00:15:56,2021-01-01 00:19:52,N,1.0,43,151,1.0,1.01,5.5,0.5,0.5,0.00,0.0,None,0.3,6.80,2.0,1.0,0.00
1,2,2021-01-01 00:25:59,2021-01-01 00:34:44,N,1.0,166,239,1.0,2.53,10.0,0.5,0.5,2.81,0.0,None,0.3,16.86,1.0,1.0,2.75
2,2,2021-01-01 00:45:57,2021-01-01 00:51:55,N,1.0,41,42,1.0,1.12,6.0,0.5,0.5,1.00,0.0,None,0.3,8.30,1.0,1.0,0.00
3,2,2020-12-31 23:57:51,2021-01-01 00:04:56,N,1.0,168,75,1.0,1.99,8.0,0.5,0.5,0.00,0.0,None,0.3,9.30,2.0,1.0,0.00
4,2,2021-01-01 00:16:36,2021-01-01 00:16:40,N,2.0,265,265,3.0,0.00,-52.0,0.0,-0.5,0.00,0.0,None,-0.3,-52.80,3.0,1.0,0.00


In [27]:
data.columns

Index(['VendorID', 'lpep_pickup_datetime', 'lpep_dropoff_datetime',
       'store_and_fwd_flag', 'RatecodeID', 'PULocationID', 'DOLocationID',
       'passenger_count', 'trip_distance', 'fare_amount', 'extra', 'mta_tax',
       'tip_amount', 'tolls_amount', 'ehail_fee', 'improvement_surcharge',
       'total_amount', 'payment_type', 'trip_type', 'congestion_surcharge'],
      dtype='object')

In [28]:
def extract_trip_time(df):
    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df['duration'] = df['duration'].apply(lambda x: x.total_seconds() / 60 ) # Time in minutes
    return df

### Q2 What's the standard deviation of the trips duration in January?

In [29]:
new_dat = extract_trip_time(data)
ans = new_dat['duration'].std()
print(f"The standard deviation of the trip duration is {ans:.2f}")

The standard deviation of the trip duration is 59.34


In [30]:
def remove_outlier(df):
    df = df[(df['duration'] >= 1) & (df['duration'] <= 60)]
    return df

### Q3 What fraction of the records left after you dropped the outliers?

In [31]:
data_without_outlier = remove_outlier(new_dat)
df_num_rows = new_dat.shape[0]
df_less_outlier_num_rows = data_without_outlier.shape[0]
percentage_left = (df_less_outlier_num_rows/df_num_rows)*100
print(f"The fraction of the records left after I dropped the outliers is {percentage_left:.2f}")

The fraction of the records left after I dropped the outliers is 96.59


In [32]:
category_columns = ['PULocationID', 'DOLocationID']

In [33]:
train_data = remove_outlier(extract_trip_time(data))
test_data = remove_outlier(extract_trip_time(val))

In [34]:
train_dicts = train_data[category_columns].fillna(-1).astype('int').astype('str').to_dict(orient="records")
val_dicts = test_data[category_columns].fillna(-1).astype('int').astype('str').to_dict(orient="records")

In [35]:
dv = DictVectorizer()
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

In [36]:
X_train.shape, X_val.shape

((73908, 506), (61921, 506))

### Q4 What's the dimensionality of this matrix (number of columns)?

In [37]:
print(f"The dimensionality of this matrix is {X_train.shape[1]}")

The dimensionality of this matrix is 506


In [38]:

y_train = train_data["duration"].values
y_val = test_data["duration"].values

In [39]:
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred = lr.predict(X_train)

### Q5 What's the RMSE on train?

In [40]:
err = mean_squared_error(y_pred, y_train, squared=False)
err
print(f"The RMSE of on the train data is {err:.2f} ")

The RMSE of on the train data is 9.78 


### Q6 What's the RMSE on validation?

In [41]:
y_pred_1 = lr.predict(X_val)
err_1= mean_squared_error(y_pred_1, y_val, squared=False)
err_1
print(f"The RMSE on the validation data is {err_1:.2f}")

The RMSE on the validation data is 10.47


In [43]:
with open('models/lin_reg.bin', 'wb') as f_out:
    pickle.dump((dv, lr), f_out)

In [46]:
with mlflow.start_run():
    mlflow.set_tag("developer", "muhammed")
    mlflow.log_param("train_data", "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-01.parquet")
    mlflow.log_param("val_data", "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-02.parquet")
    alpha = 0.01
    mlflow.log_param("alpha", alpha)
    las = Lasso(alpha)
    las.fit(X_train, y_train)
    y_pred = las.predict(X_train)

    err = mean_squared_error(y_pred, y_train, squared=False)
    print(f"The RMSE of on the train data is {err:.2f} ")
    mlflow.log_metric("rmse", err)

The RMSE of on the train data is 10.18 


In [51]:
import xgboost as xgb

from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope

In [52]:
train = xgb.DMatrix(X_train, label=y_train)
valid = xgb.DMatrix(X_val, label=y_val)

In [53]:
def objective(params):
    with mlflow.start_run():
        mlflow.set_tag("model", "xgboost")
        mlflow.log_params(params)
        booster = xgb.train(
            params=params,
            dtrain=train,
            num_boost_round=1000,
            evals=[(valid, 'validation')],
            early_stopping_rounds=50
        )
        y_pred = booster.predict(valid)
        rmse = mean_squared_error(y_val, y_pred, squared=False)
        mlflow.log_metric("rmse", rmse)

    return {'loss': rmse, 'status': STATUS_OK}

In [54]:
search_space = {
    'max_depth': scope.int(hp.quniform('max_depth', 4, 100, 1)),
    'learning_rate': hp.loguniform('learning_rate', -3, 0),
    'reg_alpha': hp.loguniform('reg_alpha', -5, -1),
    'reg_lambda': hp.loguniform('reg_lambda', -6, -1),
    'min_child_weight': hp.loguniform('min_child_weight', -1, 3),
    'objective': 'reg:linear',
    'seed': 42
}

best_result = fmin(
    fn=objective,
    space=search_space,
    algo=tpe.suggest,
    max_evals=50,
    trials=Trials()
)

  0%|          | 0/50 [00:00<?, ?trial/s, best loss=?]

[11:49:56] WARNING: /Users/runner/work/xgboost/xgboost/python-package/build/temp.macosx-10.9-x86_64-3.7/xgboost/src/objective/regression_obj.cu:203: reg:linear is now deprecated in favor of reg:squarederror.


[0]	validation-rmse:19.70712                          
[1]	validation-rmse:18.39572                          
[2]	validation-rmse:17.23537                          
[3]	validation-rmse:16.20204                          
[4]	validation-rmse:15.30215                          
[5]	validation-rmse:14.49933                          
[6]	validation-rmse:13.80024                          
[7]	validation-rmse:13.18260                          
[8]	validation-rmse:12.65225                          
[9]	validation-rmse:12.20105                          
[10]	validation-rmse:11.77810                         
[11]	validation-rmse:11.43326                         
[12]	validation-rmse:11.09537                         
[13]	validation-rmse:10.79229                         
[14]	validation-rmse:10.57349                         
[15]	validation-rmse:10.36342                         
[16]	validation-rmse:10.18472                         
[17]	validation-rmse:10.03871                         
[18]	valid

[11:56:22] WARNING: /Users/runner/work/xgboost/xgboost/python-package/build/temp.macosx-10.9-x86_64-3.7/xgboost/src/objective/regression_obj.cu:203: reg:linear is now deprecated in favor of reg:squarederror.


[1]	validation-rmse:10.23188                                                      
[2]	validation-rmse:9.53056                                                       
[3]	validation-rmse:9.24265                                                       
[4]	validation-rmse:9.06931                                                       
[5]	validation-rmse:8.98247                                                       
[6]	validation-rmse:8.92512                                                       
[7]	validation-rmse:8.67630                                                       
[8]	validation-rmse:8.61384                                                       
[9]	validation-rmse:8.57605                                                       
[10]	validation-rmse:8.54069                                                      
[11]	validation-rmse:8.49741                                                      
[12]	validation-rmse:8.46264                                                      
[13]

...

[1]	validation-rmse:11.13181                                                         
[2]	validation-rmse:10.07403                                                         
[3]	validation-rmse:9.59402                                                          
[4]	validation-rmse:9.40218                                                          
[5]	validation-rmse:9.27816                                                          
[6]	validation-rmse:9.18295                                                          
[7]	validation-rmse:9.12764                                                          
[8]	validation-rmse:9.05075                                                          
[9]	validation-rmse:8.97595                                                          
[10]	validation-rmse:8.89120                                                         
[11]	validation-rmse:8.85778                                                         
[12]	validation-rmse:8.82886                          

[14:55:36] WARNING: /Users/runner/work/xgboost/xgboost/python-package/build/temp.macosx-10.9-x86_64-3.7/xgboost/src/objective/regression_obj.cu:203: reg:linear is now deprecated in favor of reg:squarederror.


[1]	validation-rmse:9.73604                                                          
[2]	validation-rmse:9.41045                                                          
[3]	validation-rmse:9.22769                                                          
[4]	validation-rmse:9.01731                                                          
[5]	validation-rmse:8.93680                                                          
[6]	validation-rmse:8.84471                                                          
[7]	validation-rmse:8.74882                                                          
[8]	validation-rmse:8.71050                                                          
[9]	validation-rmse:8.68043                                                          
[10]	validation-rmse:8.65527                                                         
[11]	validation-rmse:8.61726                                                         
[12]	validation-rmse:8.59576                          

[14:58:39] WARNING: /Users/runner/work/xgboost/xgboost/python-package/build/temp.macosx-10.9-x86_64-3.7/xgboost/src/objective/regression_obj.cu:203: reg:linear is now deprecated in favor of reg:squarederror.


[1]	validation-rmse:15.19340                                                         
[2]	validation-rmse:13.37153                                                         
[3]	validation-rmse:12.03899                                                         
[4]	validation-rmse:11.12741                                                         
[5]	validation-rmse:10.39076                                                         
[6]	validation-rmse:9.93061                                                          
[7]	validation-rmse:9.61451                                                          
[8]	validation-rmse:9.29423                                                          
[9]	validation-rmse:9.12308                                                          
[10]	validation-rmse:8.99164                                                         
[11]	validation-rmse:8.86711                                                         
[12]	validation-rmse:8.78561                          

In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.svm import LinearSVR

mlflow.sklearn.autolog()

for model_class in (RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor, LinearSVR):

    with mlflow.start_run():

        mlflow.log_param("train-data-path", "./data/green_tripdata_2021-01.csv")
        mlflow.log_param("valid-data-path", "./data/green_tripdata_2021-02.csv")
        mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocessor")

        mlmodel = model_class()
        mlmodel.fit(X_train, y_train)

        y_pred = mlmodel.predict(X_val)
        rmse = mean_squared_error(y_val, y_pred, squared=False)
        mlflow.log_metric("rmse", rmse)